In [ ]:
# Importação de bibliotecas essenciais
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
from mpl_toolkits.mplot3d import Axes3D

# Definindo a semente para reprodutibilidade dos resultados
np.random.seed(1234)


In [ ]:
# Função para calcular a superfície de erro J(a1, a2)
def calculateErrorSurface(X, y):
    N = len(y)
    M = 200
    a1 = np.linspace(-12.0, 14.0, M)
    a2 = np.linspace(-12.0, 14.0, M)
    A1, A2 = np.meshgrid(a1, a2)

    x1 = X[:,0].reshape(N, 1)
    x2 = X[:,1].reshape(N, 1)

    J = np.zeros((M,M))
    for i in range(M):
        for j in range(M):
            yhat = A1[i,j]*x1 + A2[i,j]*x2
            J[i,j] = (1.0/N)*np.sum((y - yhat)**2)
    return J, A1, A2


In [ ]:
# Função que resolve a regressão linear pela Equação Normal
def calculateClosedFormSolution(X, y):
    N = len(y)
    a_opt = np.linalg.pinv(X.T.dot(X)).dot(X.T.dot(y))
    yhat = X.dot(a_opt)
    Joptimum = (1.0/N)*np.sum((y - yhat)**2)
    return Joptimum, a_opt


In [ ]:
# Gradiente Descendente em Batelada
def batchGradientDescent(X, y, alpha=0.0001, max_epochs=10000):
    N = len(y)
    Jgd = np.zeros(max_epochs+1)
    grad_hist = np.zeros((2, max_epochs))
    a = np.array([-10.0, -10.0]).reshape(2, 1)
    a_hist = np.zeros((2, max_epochs+1))
    a_hist[:, 0] = a.ravel()

    yhat = X.dot(a)
    Jgd[0] = (1.0/N)*np.sum((y - yhat)**2)

    error = 1
    epoch = 0
    while error > 1e-5 and epoch < max_epochs - 1:
        yhat = X.dot(a)
        gradients = -X.T.dot(y - yhat)
        a -= alpha * gradients
        a_hist[:, epoch+1] = a.ravel()
        yhat = X.dot(a)
        Jgd[epoch+1] = (1.0/N)*np.sum((y - yhat)**2)
        error = abs(Jgd[epoch] - Jgd[epoch+1])
        grad_hist[:, epoch] = gradients.ravel()
        epoch += 1
    return a, a_hist, Jgd, grad_hist, epoch


In [ ]:
# Geração de dados genéricos com duas variáveis
N = 10000
x1 = np.random.randn(N, 1)
x2 = np.random.randn(N, 1)

# Função verdadeira linear
y = 1.5 * x1 + 2.5 * x2

# Adiciona ruído gaussiano
w = np.random.randn(N, 1)
y_noisy = y + w

# Matriz de atributos
X = np.c_[x1, x2]


In [ ]:
# Visualização da superfície de erro
J, A1, A2 = calculateErrorSurface(X, y_noisy)

fig = plt.figure(figsize=(7, 6))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(A1, A2, J, cmap=cm.coolwarm)
ax.set_xlabel("a1")
ax.set_ylabel("a2")
ax.set_zlabel("Erro")
ax.set_title("Superfície de Erro J(a1, a2)")
plt.show()


In [ ]:
# Solução pela Equação Normal
Jopt, a_opt = calculateClosedFormSolution(X, y_noisy)
print("Solução ótima (Equação normal):")
print(f"a1 = {a_opt[0, 0]:.4f}, a2 = {a_opt[1, 0]:.4f}")
print(f"Erro ótimo = {Jopt:.6f}")


In [ ]:
# Treinamento com Gradiente Descendente
alpha = 0.0001
max_epochs = 10000
a, a_hist, Jgd, grad_hist, epoch = batchGradientDescent(X, y_noisy, alpha, max_epochs)

print("Solução via Gradiente Descendente:")
print(f"a1 = {a[0, 0]:.4f}, a2 = {a[1, 0]:.4f}")
print(f"Erro GD final = {Jgd[epoch]:.6f}")


In [ ]:
# Contorno da função de erro e caminho do GD
plt.figure(figsize=(6, 5))
cp = plt.contour(A1, A2, J)
plt.clabel(cp)
plt.plot(a_opt[0], a_opt[1], 'r*', markersize=12, label='Solução ótima')
plt.plot(a_hist[0, :epoch], a_hist[1, :epoch], 'kx', label='GD')
plt.xlabel("a1")
plt.ylabel("a2")
plt.legend()
plt.title("Trajetória do Gradiente Descendente")
plt.grid()
plt.show()


In [ ]:
# Curva de convergência do erro
plt.figure(figsize=(6, 4))
plt.plot(np.arange(0, epoch), Jgd[:epoch])
plt.yscale("log")
plt.xlabel("Época")
plt.ylabel("Erro (MSE)")
plt.title("Convergência do Gradiente Descendente")
plt.grid()
plt.show()


In [ ]:
# Gradientes ao longo das épocas
plt.figure(figsize=(6, 4))
plt.plot(np.arange(0, epoch), grad_hist[0, :epoch], label='∂J/∂a1')
plt.plot(np.arange(0, epoch), grad_hist[1, :epoch], label='∂J/∂a2')
plt.xlabel("Época")
plt.ylabel("Gradiente")
plt.legend()
plt.title("Evolução dos Gradientes")
plt.grid()
plt.show()
